# Task 4 matched fine-tuning comparison
Attach FS2K and the complete reconstruction_continued backup as private datasets. Two fresh five-epoch arms initialize the same reconstruction generator and reset Adam. Original checkpoints remain unchanged. Run cells in order; download backup even if a time budget stops an arm.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
candidates = [p.parent for p in Path('/kaggle/input').rglob('anno_train.json') if (p.parent/'photo').is_dir() and (p.parent/'sketch').is_dir()]
print(candidates)
assert len(candidates)==1, 'Attach one FS2K dataset or set raw to its exact root.'
raw = candidates[0]
subprocess.run([sys.executable,'-m','scripts.prepare_fs2k','--raw-dir',str(raw)],cwd=repo,check=True)


In [ ]:
candidates = [p for p in Path('/kaggle/input').rglob('best.pt') if 'mlruns' not in p.parts]
for p in candidates: print(p)


In [ ]:
# Replace with the exact reconstruction checkpoint path printed above.
initial = Path('/kaggle/input/REPLACE/reconstruction/best.pt')
assert initial.is_file(), 'Set the reconstruction best.pt path'
saved = torch.load(initial, map_location='cpu', weights_only=True)
assert saved['config'].get('objective') == 'reconstruction_only'
print('Initialization epoch:', saved['progress']['epoch'])
assert saved['progress']['epoch'] == 14, 'Expected selected reconstruction epoch 14'
del saved


In [ ]:
import shutil
root = Path('/kaggle/working/task4-finetune-comparison')
restore_folder = None  # After reset: complete prior comparison backup folder.
if restore_folder is not None and not root.exists():
    shutil.copytree(Path(restore_folder), root)
for arm in ['control', 'gan']:
    output = root / arm
    command = [sys.executable, '-u', '-m', 'training.train_task4_finetune',
        '--config', str(repo / f'configs/task4_finetune_{arm}.yaml'),
        '--init-checkpoint', str(initial), '--raw-dir', str(raw),
        '--output-dir', str(output), '--device', 'cuda', '--max-hours', '1']
    if (output/'last.pt').exists(): command += ['--resume',str(output/'last.pt')]
    subprocess.run(command,cwd=repo,check=True)


In [ ]:
import json
from IPython.display import display, FileLink
for arm in ['control','gan']:
    path=root/arm/'history.json'
    if path.exists():
        history=json.loads(path.read_text())
        print(arm, [(r['epoch'],r['val_l1'],r['val_ssim']) for r in history])
backup=shutil.make_archive('/kaggle/working/task4-finetune-comparison_backup','zip',root_dir=root.parent,base_dir=root.name)
display(FileLink(backup))
